# EXP-2026-008 / Q5-E — PREP P1 + P2: 등록 자산 신원 확인

**이 노트북은 미실행 상태로 커밋된다.** 모든 출력이 비어 있고 `execution_count`
는 전부 `null` 이다.

## 무엇을 하는가

Q5-E 실행을 막고 있는 세 항목 중 **둘** 을 확정한다. P3(source-matching
differential)는 이번 범위가 **아니다**.

| | 대상 | 지금 상태 |
|---|---|---|
| **P1** | MIT-BIH publisher tree 전체 64-hex aggregate | 미등록(절단형 `0b46a411…` 만 존재) |
| **P2** | canonical Q5-D bundle 5파일 SHA-256 + subset fold | 미등록 |

## 경계 — 이 노트북이 절대 하지 않는 것

`detect_r()` 실행, M0~M4 집계, beat join 재실행, DS2 per-beat label 접근,
V10 probability 접근, association·S PR-AUC 계산, 모델 학습, Drive 파일의
이동·삭제·덮어쓰기. **파일은 바이트 단위로 해시만 하고 내용을 집계하지 않는다.**

## 이 노트북은 아무것도 등록하지 않는다

실행하면 `registration_candidates.json` 에 **관측값** 을 담을 뿐이다. 소스나
명세를 자동으로 고치지 않는다. 등록은 Codex 인수검사 뒤 **별도 결과 인수 PR**
에서만 이루어진다.

## P1 과 P2 는 독립이다

하나가 실패해도 다른 하나의 판정을 덮어쓰지 않는다. 각각의 상태와 첫 중단
사유를 따로 보존하며, **둘 다 통과할 때만** `PREP_P1_P2_PASS` 다. 실패를 규칙
완화로 해결하지 않는다.

In [ ]:
# 1. DESIGN_AND_BOUNDARIES — 셀 순서를 먼저 보여준다.
STAGES = [
    '1. DESIGN_AND_BOUNDARIES            이 표와 경계 선언',
    '2. ENVIRONMENT                      repo/모듈 로드와 staleness guard',
    '3. SYNTHETIC_FIXTURES               합성 fixture 로 경로 점검(등록 자산 미개봉)',
    '4. EXECUTION_APPROVAL               스위치 2개. 기본값은 닫힘',
    '5. DRIVE_AUTH_AND_FOLDER_ID_PREFLIGHT  folder id 로만 조회',
    '6. P1_MITDB_IDENTITY                147파일 tree 신원',
    '7. P2_Q5D_BUNDLE_IDENTITY           12파일 계약 + 5파일 input 신원',
    '8. COMBINED_DECISION                독립 gate 두 개를 합산',
    '9. BUNDLE_WRITE                     PREP bundle 원자적 기록',
    '10. HUMAN_READABLE_REPORT           사람이 읽는 요약과 다음 행동',
]
for line in STAGES:
    print(line)
print()
print('P3(source-matching differential)는 이번 범위가 아니다.')
print('이 노트북은 값을 등록하지 않는다. 관측 후보만 남긴다.')

In [ ]:
# 2. ENVIRONMENT — 모듈을 로드하고, 쓰려는 기능이 실제로 있는지 확인한다.
#    버전 정수는 안 올리면 그만이라 무력하다. 실제 capability 를 확인한다.
REPO = '/content/repo'
import os, sys, json
if not os.path.exists(REPO):
    REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO, 'mit-bih'))

import q5d_order_preserving_beat_join as BJ
import q5e_leg2_failure_mechanism_audit as Q5E
import q5e_prep_p1_p2_asset_identity as P

MISSING = [n for n in P.module_capabilities() if not hasattr(P, n)]
assert not MISSING, f'stale PREP clone, missing {MISSING}'
assert BJ.rule_fingerprint() == Q5E.REGISTERED_RULE_FINGERPRINT, \
    'frozen Q5-D rule fingerprint moved'

print('PREP module :', P.__file__)
print('Q5-E module :', Q5E.__file__)
print('frozen Q5-D :', BJ.__file__)
print()
print(P.design_card())

In [ ]:
# 3. SYNTHETIC_FIXTURES — 등록 자산을 열기 전에 합성 fixture 로 경로를 점검한다.
#    여기서 실패하면 실행 승인을 켜기 전에 고치는 편이 훨씬 싸다.
#    이 셀은 등록 자산을 열지 않고 Drive API 도 호출하지 않는다.
import subprocess
TEST = os.path.join(REPO, 'mit-bih', 'test_q5e_prep_p1_p2_asset_identity.py')
print(subprocess.run([sys.executable, TEST], capture_output=True,
                     text=True).stdout.strip() or '(테스트 출력 없음)')
print()
print('합성 fixture 는 FakeDriveAdapter 를 쓴다. 실제 Drive API 는 호출되지')
print('않고, 합성 결과 bundle 은 SYNTHETIC_FIXTURE 로 각인되어 ingestable=false')
print('가 된다. 즉 정식 결과로 승격될 수 없다.')

In [ ]:
# 4. EXECUTION_APPROVAL — 스위치 2개. 둘 다 켜야 등록 자산에 닿는다.
#    기본값은 닫힘이고, 이 노트북은 닫힌 상태로 커밋된다.
#    "한번 돌려보자"고 여기를 고치지 마라.
MODE_NOTE = 'read-only asset identity preflight (P1 + P2)'
APPROVAL = None                 # 실행 승인 토큰. 승인 전까지 None
OPEN_REGISTERED_DATA = False    # 등록 자산 개봉 스위치. 기본 False

print('mode                :', MODE_NOTE)
print('approval present    :', P.execution_is_approved(APPROVAL))
print('OPEN_REGISTERED_DATA:', OPEN_REGISTERED_DATA)
print()
print('둘 중 하나만 열려 있어도 거부된다. 그리고 그 뒤에 terminal guard 가')
print('한 번 더 막는다 — 이번 PR 에서는 guard 를 제거하지 않는다.')
print()
print(P.APPROVAL_NOTE)

In [ ]:
# 5. DRIVE_AUTH_AND_FOLDER_ID_PREFLIGHT — **보고만 한다. 인증하지 않는다.**
#    canonical bundle 은 folder **id** 로만 고른다. 같은 이름의 폴더를 찾은
#    것은 증거가 아니다 — 이 PREP 이 존재하는 이유가 바로 그 치환이다.
#
#    인증은 이 셀에 없다. 인증을 노트북에서 하면 terminal guard 를 우회하게
#    된다: guard 는 run_prep() 안에 있는데, 셀에서 adapter 를 먼저 만들면
#    guard 가 보지도 못한 상태에서 credential 이 발급된다. 그래서 인증은
#    guard **아래**, run_prep() 안에서만 일어난다. 이 셀의 인증·service·API
#    호출 횟수는 0 이다.
MITDB_DIR = ''   # 등록 publisher tree 마운트 경로 (승인 후 채운다)
MOUNT_DIR = ''   # bundle 마운트 경로. file-id 스트림이 되면 비워도 된다
OUT_DIR = '/content/drive/MyDrive/medkos_runs'

print('registered folder id :', P.SOURCE_BUNDLE_FOLDER_ID)
print('registered run       :', P.SOURCE_BUNDLE_RUN)
print('directory contract   :', len(BJ.BUNDLE_FILES), 'files')
print('Q5-E input identity  :', len(P.BUNDLE_INPUT_FILES), 'files')
print()

# 의존성은 보고만 한다. 조용한 최신판 설치는 신원 확인 도중 런타임을 바꾼다.
DEPS = P.check_runtime_dependencies()
for name, why in sorted(P.RUNTIME_DEPENDENCIES.items()):
    print(f'{name:20s} {why}')
print('missing              :', DEPS['missing'])
print()

print('인증·scope 계약 (이 셀은 계약을 설명만 한다):')
print('  요청 scope        :', P.DRIVE_READONLY_SCOPE)
print('  credential 은 run_prep() 안, terminal guard 통과 후에만 발급된다')
print('  정확히 이 scope 하나가 아니면', P.READONLY_SCOPE_UNPROVEN, '로 중단한다')
print('  더 넓은 scope 는 "이 scope 를 포함한다"는 이유로 통과되지 않는다')
print('  credential/token/authorization 은 bundle 에 절대 기록하지 않는다')
print('  이 셀에서 수행한 인증 호출 : 0')
print('  이 셀에서 수행한 API 호출  : 0')
print()

print('production 실행 순서 (셀 9 의 run_prep() 안에서 이 순서로 일어난다):')
for step in ('OPEN_REGISTERED_DATA 스위치',
             'PREP 실행 승인 토큰',
             '등록 folder id 확인',
             'terminal execution guard  ← 이번 PR 은 여기서 멈춘다',
             'dependency 확인',
             'credential 발급 + read-only scope 증명',
             'Drive v3 service / adapter 생성',
             'P1/P2 reader · Drive API 접근'):
    print('  -', step)
print()
print('bridge 규칙: file-id 스트림이면 그것으로 끝난다(마운트 불필요).')
print('마운트를 쓰면 folder-id inventory 와 exact name/size + 제공된 모든')
print('provider checksum 으로 연결을 증명해야 하고, 실패하면')
print('P2_FOLDER_ID_BRIDGE_UNRESOLVED 로 중단한다.')
print('폴더 이름 일치는 bridge 를 대신하지 못한다.')


In [ ]:
# 6. P1_MITDB_IDENTITY — gate 순서와 중단 규칙을 먼저 보여준다.
#    aggregate 는 앞의 세 gate 가 모두 통과한 뒤에만 계산된다. 실패한 tree 가
#    등록 후보로 오해될 숫자를 내놓지 않게 하려는 것이다.
for index, gate in enumerate(P.P1_GATE_ORDER, 1):
    print(f'  {index}. {gate}')
print()
print('expected files        :', len(BJ.mitdb_expected_files()))
print('publisher-listed      :', P.MITDB_PUBLISHER_LISTED_FILES,
      '(+1 = SHA256SUMS.txt 자체 → 147/147)')
print('checksum file digest  :', P.MITDB_CHECKSUM_FILE_SHA256)
print('registered prefix     :', P.MITDB_REGISTERED_AGGREGATE_PREFIX)
print()
print('147/147 의 뜻: publisher list 는 자기 자신을 검증할 수 없으므로')
print('146개는 목록으로, SHA256SUMS.txt 자체는 별도 등록 digest 로 확인한다.')
print('중단 사유: P1_FILE_SET_MISMATCH · P1_MITDB_CHECKSUM_FILE_MISMATCH ·')
print('P1_MITDB_PUBLISHER_CHECKSUM_MISMATCH · MITDB_IDENTITY_DIVERGED')

In [ ]:
# 7. P2_Q5D_BUNDLE_IDENTITY — 두 계약을 분리해서 확인한다.
for index, gate in enumerate(P.P2_GATE_ORDER, 1):
    print(f'  {index}. {gate}')
print()
print('A. directory contract : BJ.BUNDLE_FILES', len(BJ.BUNDLE_FILES), '개 완전성')
print('   missing/unexpected 0 · SUPERSEDED 부재 · code SHA · rule fingerprint')
print('B. input identity     : Q5-E 가 읽는', len(P.BUNDLE_INPUT_FILES), '개')
print('   각 파일 name/bytes/SHA-256 + subset fold')
print()
print('나머지', len(BJ.BUNDLE_FILES) - len(P.BUNDLE_INPUT_FILES),
      '개는 directory contract 소속이며 input identity 에서')
print('unexpected 로 취급되지 않는다. 이 혼동이 예전에 정상 bundle 을 거부했다.')
print()
print('중단 사유: P2_INVENTORY_AMBIGUOUS(중복 이름·하위 폴더·shortcut·trashed) ·')
print('P2_DIRECTORY_CONTRACT_FAILED · P2_SUPERSEDED_BUNDLE ·')
print('P2_FOLDER_ID_BRIDGE_UNRESOLVED · P2_MANIFEST_IDENTITY_MISMATCH')

In [ ]:
# 8. COMBINED_DECISION — 두 gate 는 과학적으로 독립이다.
print('가능한 종합 판정:')
for status in P.PREP_STATUSES:
    print('  ', status)
print()
print('규칙:')
print('  - 둘 다 통과해야만 PREP_P1_P2_PASS')
print('  - 하나가 실패해도 다른 하나의 판정을 덮어쓰지 않는다')
print('  - 둘 다 실패하면 MULTIPLE_PREP_FAILURES 이고 각 first_failure 를 보존')
print('  - 하나라도 실패하면 registration_allowed=false 다. 다만 통과한 쪽의')
print('    **관측값은 그대로 보고**한다 — 지우면 감사 증거가 사라진다.')
print('    보류되는 것은 관측이 아니라 eligible_for_registration 이다.')


In [ ]:
# 9. BUNDLE_WRITE — 실제 실행 경로. 승인 두 개가 모두 있어야 여기까지 온다.
#    지금 상태로는 terminal guard 에서 거부되며, 그것이 커밋된 의도다.
#    adapter=None 이다: 인증과 adapter 생성은 guard 아래 run_prep() 안에서만
#    일어난다. 노트북은 credential 을 만지지 않는다.
RESULT = None
if P.execution_is_approved(APPROVAL) and OPEN_REGISTERED_DATA:
    RESULT = P.run_prep(
        MITDB_DIR, P.SOURCE_BUNDLE_FOLDER_ID, OUT_DIR,
        adapter=None,              # 인증은 terminal guard 아래에서만
        mount_dir=MOUNT_DIR or None,
        approval=APPROVAL,
        open_registered_data=OPEN_REGISTERED_DATA,
        timestamp=Q5E.run_timestamp())
    print('combined :', RESULT['combined']['status'])
    print('bundle   :', RESULT['bundle']['directory'])
else:
    print('실행하지 않았다. 위의 어떤 출력도 측정값이 아니다.')
    print('인증 호출 0회 · Drive API 호출 0회 · 등록 자산 개봉 0건.')
    print()
    print('production bundle 파일 계약:')
    for name in P.bundle_files(False):
        print('  ', name)
    print()
    print('payload fold 대상(실행 종류별):')
    print('  production :', ', '.join(P.payload_files(False)))
    print('  synthetic  :', ', '.join(P.payload_files(True)))
    print()
    print('manifest.json 은 자기가 기록하는 payload fold 에서 제외된다.')
    print('manifest 자체 SHA-256 은 bundle 밖에서 동결한다 — 아래 셀 10 이')
    print('출력하고, 그 출력이 외부 동결의 1차 증거가 된다.')
    print()
    print('publish 방식: rename 이 아니라 commit marker 다.')
    print(' ', P.COMMIT_MARKER, '가 없는 디렉터리는 bundle 이 아니라 미완성')
    print('  기록이며, verify_published_bundle() 이 거부한다.')
    print('  (Drive FUSE 마운트에서 원자적 no-replace rename 을 보장할 수')
    print('   없어서 atomic-directory-publication 주장을 철회했다.)')


In [ ]:
# 10. HUMAN_READABLE_REPORT — PASS/STOP 표, 외부 동결용 digest, 다음 행동.
#     이 셀의 **저장된 출력** 이 manifest digest 외부 동결의 1차 증거다.
#     bundle 안에는 자기 digest 를 적지 않으므로, 여기가 그 값이 남는 곳이다.
def _gate_table(result_leg, order):
    """gate 별 PASS/STOP. 도달하지 못한 gate 는 '미도달'이지 통과가 아니다."""
    seen = {g['gate']: g for g in result_leg.get('gates', ())}
    print('| gate | 결과 |')
    print('|---|---|')
    for gate in order:
        entry = seen.get(gate)
        if entry is None:
            print(f'| {gate} | (미도달) |')
        else:
            print(f"| {gate} | {'PASS' if entry['ok'] else 'STOP'} |")


def report(result):
    if result is None:
        print('| leg | 상태 | 첫 중단 사유 |')
        print('|---|---|---|')
        print('| P1 | (미실행) | - |')
        print('| P2 | (미실행) | - |')
        print()
        print('bundle 파일 계약 :', ', '.join(P.bundle_files(False)))
        print('publish 표식      :', P.COMMIT_MARKER,
              '(이게 없으면 bundle 이 아니다)')
        print()
        print('P1 gate 순서 :', ' -> '.join(P.P1_GATE_ORDER))
        print('P2 gate 순서 :', ' -> '.join(P.P2_GATE_ORDER))
        print()
        print('다음 행동: 사용자 read-only 실행 승인을 받은 뒤 셀 4 의 스위치')
        print('두 개를 켜고, 셀 5 의 경로를 채운 다음 다시 실행한다.')
        return
    combined, p1, p2 = (result['combined'], result['p1'], result['p2'])
    bundle = result['bundle']

    print('## 1. 종합')
    print('| leg | 상태 | 첫 중단 사유 |')
    print('|---|---|---|')
    print(f"| P1 | {p1['status']} | {p1['first_failure']} |")
    print(f"| P2 | {p2['status']} | {p2['first_failure']} |")
    print(f"| 종합 | {combined['status']} | - |")
    print()

    print('## 2. P1 gate 별 결과')
    _gate_table(p1, P.P1_GATE_ORDER)
    gates = {g['gate']: g for g in p1.get('gates', ())}
    digest = gates.get('checksum_file_digest')
    if digest:
        print()
        print('SHA256SUMS.txt 자체 digest :',
              'PASS' if digest['ok'] else 'STOP')
        print('  observed  :', digest['observed'])
        print('  registered:', digest['registered'])
        print('  이 파일 읽은 횟수:', digest.get('reads_of_checksum_file'))
    publisher = gates.get('publisher_checksums')
    if publisher:
        print()
        print('publisher list 대조 (146 + 1 = 147/147 의 146 쪽):')
        print('  목록 제공 여부 :', publisher['available'])
        print(f"  checked/matched: {publisher['checked']}/"
              f"{publisher['matched']}"
              f" (기대 {publisher['expected_checked']})")
        print('  mismatched     :', publisher['n_mismatched'])
        print('  unlisted       :', publisher['n_unlisted'])
        for row in publisher.get('mismatched', ())[:10]:
            print(f"    - {row['name']}: published={row['published_sha256']}"
                  f" observed={row['observed_sha256']}"
                  f" crlf={row.get('has_crlf')} bom={row.get('starts_with_bom')}")
    print()
    print('147/147 = publisher list 146 개 + SHA256SUMS.txt 자체 digest 1 개.')
    print('checksum 파일은 자기 자신을 검증할 수 없으므로 따로 등록되어 있다.')
    print('per-file 관측 수 :', len(p1.get('per_file') or ()))
    print('tree aggregate   :', p1.get('tree_aggregate'))
    print('observation_only :', p1.get('observation_only'))
    print('blocked_by       :', p1.get('blocked_by'))
    print()

    print('## 3. P2 gate 별 결과')
    _gate_table(p2, P.P2_GATE_ORDER)
    print()
    print('inventory 방식   : folder **id** 직접 조회 (이름 검색 아님)')
    print('  folder id      :', p2.get('folder_id'))
    print('  children 수    :', len(p2.get('inventory') or ()))
    bridge = p2.get('bridge') or {}
    print('bridge 방식      :', bridge.get('method'))
    print('  cross-check 수 :', len(bridge.get('cross_check') or ()))
    print('  provider checksum 미제공 파일 수 :',
          bridge.get('n_without_checksum'))
    for row in (bridge.get('cross_check') or ()):
        print(f"    - {row['name']}: bytes_match={row['bytes_match']}"
              f" sha256_available={row['sha256_available']}"
              f" sha256_match={row['sha256_match']}"
              f" md5_available={row['md5_available']}"
              f" md5_match={row['md5_match']}")
    print('  (md5 는 전송 대조용이며 보안 신원이 아니다)')
    print('input_identity   :', 'None (해당 gate 미도달)'
          if p2.get('input_identity') is None else 'present')
    print('observation_only :', p2.get('observation_only'))
    print('blocked_by       :', p2.get('blocked_by'))
    print()

    print('## 4. 인증 / scope 증명')
    with open(f"{bundle['directory']}/config.json") as fh:
        cfg = _json.load(fh)
    auth = cfg['drive_authentication']
    print('  요청 scope                    :', auth.get('requested_scopes'))
    print('  관측 scope                    :', auth.get('observed_scopes'))
    print('  read-only scope 증명됨        :',
          auth.get('exact_readonly_scope_proven'))
    print('  credential 종류               :', auth.get('credential_type'))
    print('  credential 기록 여부          :', auth.get('credential_recorded'))
    print('  adapter 에 write 메서드 없음  :',
          auth.get('no_write_adapter_methods'))
    print('  사유                          :', auth.get('reason'))
    print()
    print('  runtime :', cfg['runtime']['python_version'],
          '/', cfg['runtime']['platform'])
    for dist, version in sorted(cfg['runtime']['distributions'].items()):
        print(f'    {dist:28s} {version}')
    print('    PREP module sha256          :',
          cfg['runtime']['prep_module_sha256'])
    print('    frozen Q5-D module sha256   :',
          cfg['runtime']['frozen_q5d_module_sha256'])
    print()

    print('## 5. commit / consumer 검증')
    # 이 실행 자신의 검증은 방금 계산한 digest 로 하는 자기일관성 검사다.
    # 그래서 anchor source 를 same_run_self_check 로 밝히고, 그 결과
    # manifest_anchored_externally=False · acceptance_eligible=False 가 된다.
    verified = result.get('verified') or P.verify_published_bundle(
        bundle['directory'],
        expected_manifest_sha256=bundle['manifest_sha256_freeze_externally'],
        manifest_anchor_source=P.ANCHOR_SAME_RUN)
    print('  committed             :', verified['committed'],
          f"({P.COMMIT_MARKER})")
    print('  구조 검증(structure_ok):', verified['structure_ok'])
    print('  synthetic / ingestable :', verified['synthetic_fixture'],
          '/', verified['ingestable'],
          '(둘 다 JSON boolean 이어야 하며 서로 부정 관계)')
    print('  payload fold 재계산    :', verified['prep_payload_sha256'])
    print('    (marker 를 믿지 않고 고정 코드 계약으로 재계산한 값이며,')
    print('     marker fold · manifest fold 와 모두 일치해야 한다)')
    print('  manifest digest 일치   :',
          verified['manifest_digest_matches_expected'])
    print('  anchor 출처            :', verified['manifest_anchor_source'])
    print('  외부 anchor 여부       :', verified['manifest_anchored_externally'])
    print('  acceptance 자격        :', verified['acceptance_eligible'])
    print('  판정 근거              :', verified['acceptance_note'])
    print('  문제                  :', verified['problems'] or '없음')
    print()
    print('  digest 가 일치하는 것과 anchor 되었다는 것은 다른 사실이다.')
    print('  이 실행이 자기가 방금 계산한 값과 비교한 것은 자기일관성 검사이지')
    print('  "그 뒤로 편집되지 않았다"는 증거가 아니다 — 아직 "그 뒤"가 없다.')
    print(f"  외부로 인정되는 출처는 {list(P.EXTERNAL_MANIFEST_ANCHORS)} 뿐이며,")
    print('  검토자는 아래 manifest digest 를 이 셀의 저장된 출력(또는 등록')
    print('  기록)에서 가져와 다음처럼 넣어야 acceptance 판정이 난다:')
    print('    P.verify_published_bundle(dir,')
    print('        expected_manifest_sha256=<저장된 출력의 값>,')
    print(f"        manifest_anchor_source=P.ANCHOR_SAVED_NOTEBOOK)")
    print('  구조 검증 통과는 acceptance PASS 가 아니다.')
    print('  marker 가 없으면 bundle 이 아니다 — rename 이 아니라 이 표식이')
    print('  publish 의 정의다. 실패한 실행은 marker 없는 디렉터리를 그대로')
    print('  남기며, 아무것도 삭제하지 않는다.')
    print()

    print('## 6. 외부 동결 대상 (전체 64-hex)')
    print('prep_payload_sha256              :', bundle['prep_payload_sha256'])
    print('manifest_sha256_freeze_externally:',
          bundle['manifest_sha256_freeze_externally'])
    print('payload_files                    :', bundle['payload_files'])
    print()

    print('## 7. 등록 후보 — 관측과 자격은 별개다')
    print('registration_allowed :', combined['registration_allowed'])
    with open(f"{bundle['directory']}/registration_candidates.json") as fh:
        candidates = _json.load(fh)
    for key in ('MITDB_TREE_AGGREGATE', 'SOURCE_BUNDLE_FILE_SHA256'):
        entry = candidates[key]
        print(f"  {key}")
        print(f"    gate_passed              : {entry['gate_passed']}")
        print(f"    eligible_for_registration: "
              f"{entry['eligible_for_registration']}")
        print(f"    applied_automatically    : "
              f"{entry['applied_automatically']}")
        print(f"    blocked_by               : {entry['blocked_by']}")
        print(f"    observed                 : {entry['observed']}")
        print(f"    observation_note         : {entry['observation_note']}")
    print()

    print('## 8. 다음 행동')
    if combined['status'] == P.PREP_PASS:
        print('위 두 digest 를 실행계약 Decision log 와 등록 기록에 그대로')
        print('옮겨 적고, bundle 을 Codex 결과 인수에 제출한다. 값 등록은')
        print('별도 결과 인수 PR 에서만 이루어진다 — 이 노트북은 등록하지 않는다.')
    else:
        print('중단 사유:', combined['status'])
        print('실패를 규칙 완화로 해결하지 않는다. 위 gate 표에서 STOP 인')
        print('지점의 관측값을 그대로 Codex 에 제출하고 판단을 받는다.')


import json as _json
report(RESULT)


## 이 노트북이 해서는 안 되는 것

등록 자산 수정·이동·삭제, `detect_r()` 실행, M0~M4 집계, beat join 재실행,
DS2 per-beat label 접근, V10 probability 접근, association·S PR-AUC 계산,
모델 학습, 기존 run bundle·null shard 변경.

파일은 **바이트 해시만** 한다. parquet 을 파싱하거나 내용을 집계하지 않는다.

## 다음 단계

1. Codex 구현 인수검사
2. 사용자 read-only 실행 승인
3. P1·P2 실행 및 bundle 보존
4. Codex 결과 인수
5. 별도 결과 인수 PR 에서 값 등록
6. P3 승인·구현·실행
7. **그 뒤에만** Q5-E 실행 승인 여부를 판단한다